# Sketch2Shape AI — live demo on a Kaggle T4

1. **Settings → Accelerator: GPU T4**, **Internet: on**.
2. Add the dataset **`sketch2shape-src`** (or clone the repo in cell 1).
3. *Run all.* Setup ≈ 7 min + weights ≈ 25 GB; the last cell prints a public `*.gradio.live` link.

Every number in the README was produced by the same scripts (`kaggle/common.sh`, `scripts/setup_backend.sh`).

In [ ]:
import glob, os, shutil, subprocess
# Option A: attached Kaggle dataset; Option B: git clone (edit the URL)
src = glob.glob('/kaggle/input/**/kaggle/tasks.sh', recursive=True)
if src:
    shutil.copytree(os.path.dirname(os.path.dirname(src[0])), '/tmp/s2s/repo', dirs_exist_ok=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AshmitGarg69/sketch2shape', '/tmp/s2s/repo'], check=True)
%cd /tmp/s2s/repo

In [ ]:
%%bash
source kaggle/common.sh
setup_backend triposr > $OUT/logs/b_triposr.txt 2>&1 &
setup_backend instantmesh > $OUT/logs/b_instantmesh.txt 2>&1 &
setup_parent
weights balanced triposr,instantmesh
wait
cat $OUT/logs/b_*.txt
.venv/bin/python -m sketch2shape.cli doctor

In [ ]:
# Launch the UI with a public link (keep this cell running during the demo)
import os, subprocess
env = dict(os.environ, S2S_WEIGHTS='/tmp/s2s/weights', HF_HOME='/tmp/s2s/weights/hf', TORCH_HOME='/tmp/s2s/weights/torch',
           U2NET_HOME='/tmp/s2s/weights/u2net', CUDA_VISIBLE_DEVICES='0', HF_HUB_OFFLINE='1')
proc = subprocess.Popen(['.venv/bin/python', '-m', 'sketch2shape.cli', 'serve', '--share'], env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
    if 'gradio.live' in line:
        print('\n>>> Open the link above. Scripted use: sketch2shape run --remote <link> --sketch my.png')